# ReWOO [Step 08.04 - Reasoning WithOut Observation]

> **MLCourse - Agentic AI - LangGraph**

ReWOO (Xu et al., 2023) asks a sharp question:

> *Why does the reasoning model need to see the tool outputs at all?*

In ReAct, every observation is appended to the context and resent on every
subsequent step. In Plan-and-Execute, every step costs a full executor LLM call.
ReWOO removes both costs with one idea: **plan with variables**.

```
  PLANNER : write every step as  #E1 = tool[args] , where later steps may
            reference earlier results BY NAME (#E1, #E2, ...)
  WORKER  : execute all the tool calls, substituting variables. NO LLM AT ALL.
  SOLVER  : ONE final LLM call, given the plan and the filled-in evidence.
```

Total LLM calls: **2**. Not two per step - two, total, regardless of how many tools run.

### What you'll learn

- The variable-substitution plan format, and how to parse it robustly.
- Why the worker needs no model, and exactly what that saves.
- The token-efficiency argument, made with **measured numbers** against ReAct.
- ReWOO's real weakness: it cannot react to what it finds.

### Key takeaways

- ReWOO is the cheapest pattern here by a wide margin on tool-heavy tasks.
- Its LLM cost is **constant** in the number of tool calls; ReAct's is quadratic.
- It fails on tasks where step 3 depends on *interpreting* the result of step 2.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                  # environment variable access
import time                                # timing + backoff sleeps
from pathlib import Path                   # locating the track root
from dotenv import load_dotenv             # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we find the track root `03_agentic_ai`,
# then load the (gitignored) .env that lives there. Every provider-touching
# notebook in this track uses exactly this block.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

GROQ_KEY = os.getenv("GROQ_API_KEY")       # never print this value
GROQ_MODEL = "qwen/qwen3.8-27b"            # fast hosted model, generous free tier
OLLAMA_MODEL = "llama3.1:8b"               # local fallback if Groq is unavailable


def make_llm(temperature: float = 0.0, max_tokens: int = 512):
    """Return a chat model. Groq first (fast, hosted); local Ollama as fallback.

    OpenAI is never used anywhere in this course.
    """
    if GROQ_KEY:
        from langchain_groq import ChatGroq
        return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                        temperature=temperature, max_tokens=max_tokens)
    from langchain_ollama import ChatOllama
    return ChatOllama(model=OLLAMA_MODEL, temperature=temperature)


def safe_invoke(model, messages, retries: int = 4, pause: float = 1.5):
    """Invoke a chat model with exponential backoff on rate limits (HTTP 429).

    Groq's free tier allows roughly 8000 tokens per minute. Teaching notebooks
    fire many small calls in a row, so a retry loop is not optional here.
    """
    delay = pause
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(pause)              # pace the next call politely
            return out
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print("  [backoff] %s -- retrying in %.1fs" % (type(exc).__name__, delay))
            time.sleep(delay)
            delay *= 2                     # exponential backoff
    raise RuntimeError("unreachable")


print("Track root :", TRACK.name)
print("Provider   :", "Groq / " + GROQ_MODEL if GROQ_KEY else "Ollama / " + OLLAMA_MODEL)


### The shared task world


In [ ]:
# Every notebook in this module attacks THE SAME task with a different reasoning
# pattern, so the comparison in notebook 05 is apples-to-apples.

from langchain_core.tools import tool

# A tiny deterministic "database". Deterministic matters: we need to check
# correctness automatically, without a human reading the answer.
POPULATION = {"tokyo": 13_960_000, "lagos": 15_400_000, "lima": 9_750_000}
AREA_KM2 = {"tokyo": 2194, "lagos": 1171, "lima": 2672}

TOOL_CALLS = {"count": 0}          # instrumentation: how many tool calls happened


@tool
def population(city: str) -> str:
    """Return the population of a city as a plain number string.

    Args:
        city: City name, e.g. "Tokyo".
    """
    TOOL_CALLS["count"] += 1
    return str(POPULATION.get(city.strip().lower(), "unknown city"))


@tool
def area_km2(city: str) -> str:
    """Return the land area of a city in square kilometres as a plain number string.

    Args:
        city: City name, e.g. "Tokyo".
    """
    TOOL_CALLS["count"] += 1
    return str(AREA_KM2.get(city.strip().lower(), "unknown city"))


TOOLS = [population, area_km2]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

TASK = (
    "Among Tokyo, Lagos and Lima, which city has the highest population density "
    "(people per square kilometre)? Answer with the city name and the density "
    "rounded to the nearest whole number."
)

# Ground truth, computed here so the notebook can grade itself.
DENSITIES = {c: POPULATION[c] / AREA_KM2[c] for c in POPULATION}
GT_CITY = max(DENSITIES, key=DENSITIES.get)
GT_DENSITY = round(DENSITIES[GT_CITY])

print("Task:", TASK)
print()
for c in sorted(DENSITIES, key=DENSITIES.get, reverse=True):
    print("  %-6s %9d / %5d = %7.0f people/km2" % (c, POPULATION[c], AREA_KM2[c], DENSITIES[c]))
print()
print("Ground truth -> %s, %d" % (GT_CITY.title(), GT_DENSITY))


def grade(answer: str) -> bool:
    """Automatic grader: the answer must name the right city AND the right density.

    The density is accepted within +/-2 to tolerate rounding differences.
    """
    import re
    if not answer:
        return False
    low = answer.lower()
    if GT_CITY not in low:
        return False
    cleaned = low.replace(",", "").replace(".", " ")
    numbers = [int(n) for n in re.findall(r"\d+", cleaned)]
    return any(abs(n - GT_DENSITY) <= 2 for n in numbers)


### Instrumentation: a token and latency meter


In [ ]:
import time


class Meter:
    """Accumulates token usage, call counts and wall-clock time for one run.

    Every pattern in this module is wrapped in one of these, so notebook 05 can
    compare them on identical instrumentation.
    """

    def __init__(self, name):
        self.name = name
        self.input_tokens = 0
        self.output_tokens = 0
        self.llm_calls = 0
        self.tool_calls = 0
        self.seconds = 0.0
        self._t0 = None

    def start(self):
        TOOL_CALLS["count"] = 0
        self._t0 = time.time()
        return self

    def stop(self):
        self.seconds = time.time() - self._t0
        self.tool_calls = TOOL_CALLS["count"]
        return self

    def record(self, message):
        """Add one AIMessage's usage to the totals, then return the message."""
        usage = getattr(message, "usage_metadata", None) or {}
        if usage:
            self.input_tokens += usage.get("input_tokens", 0)
            self.output_tokens += usage.get("output_tokens", 0)
            self.llm_calls += 1
        return message

    def record_all(self, messages):
        """Add usage from every AIMessage in a list (for create_agent results)."""
        for m in messages:
            if getattr(m, "usage_metadata", None):
                self.record(m)
        return messages

    @property
    def total_tokens(self):
        return self.input_tokens + self.output_tokens

    def report(self, answer=None, correct=None):
        print()
        print("=" * 62)
        print("PATTERN : %s" % self.name)
        print("-" * 62)
        print("LLM calls    : %d" % self.llm_calls)
        print("tool calls   : %d" % self.tool_calls)
        print("input tokens : %d" % self.input_tokens)
        print("output tokens: %d" % self.output_tokens)
        print("TOTAL tokens : %d" % self.total_tokens)
        print("latency      : %.1fs" % self.seconds)
        if correct is not None:
            print("correct      : %s" % ("YES" if correct else "NO"))
        print("=" * 62)
        if answer:
            print(answer)
        return self


### 1. The plan format

Everything hinges on a plan the machine can execute without a model. We ask for
lines in exactly this shape:

```
Plan: <why this step exists>
### E1 = population[Tokyo]
Plan: <why this step exists>
### E2 = area_km2[Tokyo]
```

A later step may write `#E1` inside its argument and the worker substitutes the
actual value. That variable reference is the entire mechanism - it lets the plan
express data dependencies with no model in the loop.

In [4]:
from typing import TypedDict
import re
from langgraph.graph import StateGraph, START, END


class ReWOOState(TypedDict):
    task: str
    plan_text: str          # the planner's raw output, kept for the solver
    steps: list             # parsed [(rationale, var, tool, arg), ...]
    results: dict           # {"#E1": "13960000", ...}
    answer: str


# Matches:  #E1 = population[Tokyo]
PLAN_LINE = re.compile(r"#(E\d+)\s*=\s*(\w+)\s*\[([^\]]*)\]")
print("parser ready:", PLAN_LINE.pattern)

parser ready: #(E\d+)\s*=\s*(\w+)\s*\[([^\]]*)\]


### 2. The planner - the first of only two LLM calls

The prompt does a lot of work: it must produce a format strict enough to parse.
A **few-shot example is not optional** here; a plain instruction produces prose
about half the time.

In [5]:
planner_llm = make_llm(max_tokens=520)

PLANNER_PROMPT = """You write executable plans. For the task below, produce a plan
where each step is a tool call whose result is stored in a variable #E1, #E2, ...

Available tools:
  population[city]  -- returns the population of the city as a number
  area_km2[city]    -- returns the land area of the city in square kilometres

Format each step on exactly two lines:
Plan: <one short sentence saying why>
#E<n> = <tool>[<argument>]

A later step may use an earlier variable inside its argument, e.g. #E3 = tool[#E1].
Do NOT do arithmetic in the plan - only tool calls. Output the plan and nothing else.

Example for "What is the population of Paris and of Rome?":
Plan: Look up the population of Paris.
#E1 = population[Paris]
Plan: Look up the population of Rome.
#E2 = population[Rome]

Task: {task}
"""


def planner(state: ReWOOState) -> dict:
    """LLM CALL 1 of 2: produce the whole plan, with variables."""
    raw = meter.record(safe_invoke(planner_llm,
                                   PLANNER_PROMPT.format(task=state["task"]))).content

    steps, rationale = [], ""
    for line in raw.splitlines():
        line = line.strip()
        if line.lower().startswith("plan:"):
            rationale = line.split(":", 1)[1].strip()
            continue
        hit = PLAN_LINE.search(line)
        if hit and hit.group(2) in TOOLS_BY_NAME:     # ignore hallucinated tools
            steps.append((rationale, "#" + hit.group(1), hit.group(2), hit.group(3).strip()))
            rationale = ""

    print("[planner] parsed %d executable steps:" % len(steps))
    for why, var, tool_name, arg in steps:
        print("   %-4s = %-11s[%s]" % (var, tool_name, arg))
    return {"plan_text": raw, "steps": steps}

### 3. The worker - **zero LLM calls**

This is the whole point. Read this function and notice what is missing: no model,
no prompt, no tokens. It is a loop over the parsed plan doing string substitution
and calling functions.

On a task with 20 tool calls, ReAct would spend 20 LLM round trips here.
Plan-and-Execute would spend 20 executor calls *plus* 20 replanner calls. ReWOO
spends nothing.

In [6]:
def worker(state: ReWOOState) -> dict:
    """NO LLM. Pure execution with variable substitution."""
    results = {}
    for why, var, tool_name, arg in state["steps"]:
        resolved = arg
        for prev_var, prev_val in results.items():          # substitute #E1, #E2, ...
            resolved = resolved.replace(prev_var, str(prev_val))

        value = TOOLS_BY_NAME[tool_name].invoke({"city": resolved})
        results[var] = value
        print("[worker ] %-4s = %-11s[%-8s] -> %s" % (var, tool_name, resolved, value))

    return {"results": results}

### 4. The solver - the second and last LLM call

The solver sees the plan and the filled-in evidence, and produces the answer. It
never saw a tool schema, never negotiated a tool call, and never re-read a growing
transcript. It gets one compact, information-dense prompt.

In [7]:
solver_llm = make_llm(max_tokens=340)


def solver(state: ReWOOState) -> dict:
    """LLM CALL 2 of 2: one pass over the evidence."""
    # IMPORTANT: label every row. A bare "#E1 = 13960000" tells the solver
    # nothing about WHICH city that number belongs to (see the note below).
    evidence = "\n".join("%s = %s[%s] -> %s" % (v, t, a, state["results"][v])
                         for _, v, t, a in state["steps"] if v in state["results"])

    prompt = ("Task: %s\n\n"
              "A plan was executed and produced this evidence:\n%s\n\n"
              "Plan for reference:\n%s\n\n"
              "Using ONLY the evidence above, do any arithmetic needed and give the "
              "final answer in one short sentence."
              % (state["task"], evidence, state["plan_text"]))

    print("[solver ] evidence block is %d characters" % len(evidence))
    return {"answer": meter.record(safe_invoke(solver_llm, prompt)).content.strip()}

> ### Pitfall: the evidence table must be self-describing
>
> An earlier version of this notebook built the evidence block as bare rows:
>
> ```
> #E1 = 13960000
> #E2 = 2194
> #E3 = 15400000
> ```
>
> The solver saw six numbers with no labels, had to guess which number belonged to
> which city from the plan text alone, and **got the answer wrong** - it paired a
> population with the wrong area and named Tokyo instead of Lagos.
>
> This is the most common real-world ReWOO bug, and it follows directly from the
> pattern's core design. In ReAct, every observation arrives in context *right next
> to the tool call that produced it*, so the pairing is free. ReWOO deliberately
> throws that context away to save tokens - which means **you** are now responsible
> for putting the labels back. The evidence table is the only thing the solver ever
> sees.
>
> The fix is one line: emit `#E1 = population[Tokyo] -> 13960000` instead of
> `#E1 = 13960000`. It costs a few tokens per row and it is not optional.

In [8]:
wg = StateGraph(ReWOOState)
wg.add_node("planner", planner)
wg.add_node("worker", worker)
wg.add_node("solver", solver)
wg.add_edge(START, "planner")
wg.add_edge("planner", "worker")
wg.add_edge("worker", "solver")
wg.add_edge("solver", END)

rewoo_app = wg.compile()
print(rewoo_app.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | planner |   
 +---------+   
      *        
      *        
      *        
  +--------+   
  | worker |   
  +--------+   
      *        
      *        
      *        
  +--------+   
  | solver |   
  +--------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


### 5. Run it


In [9]:
meter = Meter("ReWOO").start()
out = rewoo_app.invoke({"task": TASK, "plan_text": "", "steps": [],
                        "results": {}, "answer": ""})
meter.stop()

rewoo_answer = out["answer"]
meter.report(rewoo_answer, grade(rewoo_answer))

[planner] parsed 6 executable steps:
   #E1  = population [Tokyo]
   #E2  = area_km2   [Tokyo]
   #E3  = population [Lagos]
   #E4  = area_km2   [Lagos]
   #E5  = population [Lima]
   #E6  = area_km2   [Lima]
[worker ] #E1  = population [Tokyo   ] -> 13960000
[worker ] #E2  = area_km2   [Tokyo   ] -> 2194
[worker ] #E3  = population [Lagos   ] -> 15400000
[worker ] #E4  = area_km2   [Lagos   ] -> 1171
[worker ] #E5  = population [Lima    ] -> 9750000
[worker ] #E6  = area_km2   [Lima    ] -> 2672
[solver ] evidence block is 194 characters



PATTERN : ReWOO
--------------------------------------------------------------
LLM calls    : 2
tool calls   : 6
input tokens : 575
output tokens: 144
TOTAL tokens : 719
latency      : 4.4s
correct      : YES
Lagos has the highest population density at 13,151 people per square kilometre.


In [10]:
print("RAW PLAN FROM THE PLANNER")
print(out["plan_text"])
print()
print("EVIDENCE TABLE")
for var, val in out["results"].items():
    print("  %-4s = %s" % (var, val))

RAW PLAN FROM THE PLANNER
Plan: Look up the population of Tokyo.
#E1 = population[Tokyo]
Plan: Look up the area of Tokyo.
#E2 = area_km2[Tokyo]
Plan: Look up the population of Lagos.
#E3 = population[Lagos]
Plan: Look up the area of Lagos.
#E4 = area_km2[Lagos]
Plan: Look up the population of Lima.
#E5 = population[Lima]
Plan: Look up the area of Lima.
#E6 = area_km2[Lima]

EVIDENCE TABLE
  #E1  = 13960000
  #E2  = 2194
  #E3  = 15400000
  #E4  = 1171
  #E5  = 9750000
  #E6  = 2672


### 6. The token-efficiency argument, with real numbers

Now the concrete comparison. We run the *same* task through a ReAct agent right
here, so the numbers come from the same session, model and tools.

In [11]:
from langchain.agents import create_agent

react_meter = Meter("ReAct").start()
react_agent = create_agent(
    model=make_llm(max_tokens=400), tools=TOOLS,
    system_prompt=("You are a careful analyst. Use the provided tools to look up "
                   "facts. Do the arithmetic yourself. Give a short final answer."))
react_out = react_agent.invoke({"messages": [("user", TASK)]})
react_meter.stop()
react_meter.record_all(react_out["messages"])
react_answer = react_out["messages"][-1].content
react_meter.report(react_answer, grade(react_answer))


PATTERN : ReAct
--------------------------------------------------------------
LLM calls    : 3
tool calls   : 7
input tokens : 1874
output tokens: 309
TOTAL tokens : 2183
latency      : 2.0s
correct      : YES
Tokyo: 13,960,000 / 2,194 ≈ 6,363
Lagos: 15,400,000 / 1,171 ≈ 13,151
Lima: 9,750,000 / 2,672 ≈ 3,649

Lagos has the highest population density.

**Lagos, ≈ 13,151 people/km²**


In [12]:
print("%-10s %10s %10s %10s %9s %8s" % ("pattern", "LLM calls", "in tok", "out tok", "TOTAL", "sec"))
print("-" * 62)
for m in (react_meter, meter):
    print("%-10s %10d %10d %10d %9d %8.1f"
          % (m.name, m.llm_calls, m.input_tokens, m.output_tokens, m.total_tokens, m.seconds))

saving = react_meter.total_tokens - meter.total_tokens
pct = 100 * saving / react_meter.total_tokens if react_meter.total_tokens else 0
print()
print("ReWOO used %d fewer tokens than ReAct (%.0f%% reduction)." % (saving, pct))
print("ReWOO made %d LLM calls; ReAct made %d." % (meter.llm_calls, react_meter.llm_calls))
print("ReWOO ran %d tool calls with ZERO model involvement." % len(out["results"]))

pattern     LLM calls     in tok    out tok     TOTAL      sec
--------------------------------------------------------------
ReAct               3       1874        309      2183      2.0
ReWOO               2        575        144       719      4.4

ReWOO used 1464 fewer tokens than ReAct (67% reduction).
ReWOO made 2 LLM calls; ReAct made 3.
ReWOO ran 6 tool calls with ZERO model involvement.


### Why the gap exists - and why it widens

ReAct's input cost is roughly:

```
sum over steps k of ( system + question + everything from steps 1..k-1 )
```

which is **O(N^2)** in the number of steps. ReWOO's is:

```
(planner prompt) + (solver prompt containing a compact evidence table)
```

The evidence table grows linearly, but it appears in **one** prompt rather than
being nested inside N of them. Let's project both from the measured per-call sizes.

In [13]:
react_inputs = [m.usage_metadata["input_tokens"] for m in react_out["messages"]
                if getattr(m, "usage_metadata", None)]
base = react_inputs[0]
per_step = ((react_inputs[-1] - react_inputs[0]) / (len(react_inputs) - 1)
            if len(react_inputs) > 1 else 0)

print("measured ReAct base prompt     : %d input tokens" % base)
print("measured ReAct growth per step : %.0f input tokens" % per_step)
print("measured ReWOO total input     : %d tokens" % meter.input_tokens)
print()
print("%6s %18s %18s %10s" % ("steps", "ReAct input tok", "ReWOO input tok", "ratio"))
print("-" * 56)
for n in (3, 6, 12, 25, 50):
    react_proj = sum(base + per_step * k for k in range(n))
    rewoo_proj = meter.input_tokens + 8 * n      # ~8 tokens per extra evidence row
    print("%6d %18d %18d %9.1fx" % (n, react_proj, rewoo_proj, react_proj / rewoo_proj))
print()
print("The gap is not a constant factor - it widens with every step.")

measured ReAct base prompt     : 444 input tokens
measured ReAct growth per step : 148 input tokens
measured ReWOO total input     : 575 tokens

 steps    ReAct input tok    ReWOO input tok      ratio
--------------------------------------------------------
     3               1777                599       3.0x
     6               4891                623       7.9x
    12              15129                671      22.5x
    25              55650                775      71.8x
    50             204112                975     209.3x

The gap is not a constant factor - it widens with every step.


### 7. Parallelism comes free

The worker knows every tool call before executing any of them, so independent calls
can run concurrently. ReAct structurally cannot do this, because each call waits for
the previous observation.

In [14]:
import concurrent.futures as cf

independent = [(v, t, a) for _, v, t, a in out["steps"] if "#E" not in a]
print("steps with no variable dependency: %d of %d" % (len(independent), len(out["steps"])))

t0 = time.time()
with cf.ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(TOOLS_BY_NAME[t].invoke, {"city": a}): v for v, t, a in independent}
    parallel = {futures[f]: f.result() for f in cf.as_completed(futures)}
print("executed %d tool calls in parallel in %.3fs" % (len(parallel), time.time() - t0))
print("results:", dict(sorted(parallel.items())))

steps with no variable dependency: 6 of 6
executed 6 tool calls in parallel in 0.003s
results: {'#E1': '13960000', '#E2': '2194', '#E3': '15400000', '#E4': '1171', '#E5': '9750000', '#E6': '2672'}


### 8. ReWOO's real weakness

Everything above is the upside. Here is the honest downside, and it is significant:

> **ReWOO cannot react to what it finds.** The plan is fixed before any evidence
> exists. If step 2's result should change what step 3 *does*, ReWOO has no mechanism
> for that - variable substitution passes **values**, not **decisions**.

Concretely, ReWOO struggles with:

- *"Find the largest city in the dataset, then look up its area."* - the second
  lookup's argument depends on interpreting the first result. (Substitution can pass
  a value along, but only if the tool returns exactly the right token.)
- *"Search for X; if you find nothing, search for Y instead."* - a branch.
- *"Keep querying until the total exceeds one million."* - a loop.

Let's watch the branch case fail.

In [15]:
branchy = ("Look up the population of Tokyo. If it is above 10 million, report its "
           "area; otherwise report the area of Lima instead.")

probe = planner({"task": branchy, "plan_text": "", "steps": [], "results": {}, "answer": ""})
print()
print("The planner had to COMMIT to a branch before seeing any data.")
print("Whatever it chose, it chose blind - the plan format has no conditional.")
print()
print("For tasks like this use ReAct (naturally adaptive) or Plan-and-Execute")
print("(whose replanner can revise after seeing the first result).")

[planner] parsed 3 executable steps:
   #E1  = population [Tokyo]
   #E2  = area_km2   [Tokyo]
   #E3  = area_km2   [Lima]

The planner had to COMMIT to a branch before seeing any data.
Whatever it chose, it chose blind - the plan format has no conditional.

For tasks like this use ReAct (naturally adaptive) or Plan-and-Execute
(whose replanner can revise after seeing the first result).


### 9. Choosing ReWOO

**Use it when:**

- The task is **tool-heavy** and the calls are **independent or statically ordered**.
- You know the shape of the work up front - enrich these 50 records, look up these
  12 fields, fetch these documents.
- Token cost or latency is a first-order concern.
- Your tools are reliable enough that you do not need to react to failures mid-plan.

**Do not use it when:**

- Later steps depend on **interpreting** earlier results.
- The task needs branching or looping.
- Tools fail often and recovery requires judgement.

**Hybrid worth knowing:** run ReWOO first; if the solver reports the evidence was
insufficient, fall back to ReAct for the remainder. You get ReWOO's cost on the easy
majority and ReAct's flexibility on the hard tail - and that composition is exactly
what module 07 taught you to build.

### Recap

- ReWOO = **planner with variables -> LLM-free worker -> single solver call**.
- Two LLM calls total, regardless of how many tools run.
- The cost advantage over ReAct **widens** with step count, because ReAct is quadratic.
- It buys that with rigidity: no branching, no looping, no reacting to observations.

### Next

**[05_choosing_a_pattern](05_choosing_a_pattern.ipynb)** - all four patterns, one
task, measured side by side.